In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Load the Dataset
df = pd.read_csv("ObesityDataSet_raw_and_data_sinthetic.csv")

# 2. Separate Features (X) and Target (y)
X = df.drop(columns=['NObeyesdad'])
y = df['NObeyesdad']

# 3. Encode the Target Variable to integers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 4. Identify Numerical and Categorical Columns
num_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# 5. Build the Preprocessing Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
    ])

# 6. Construct the Pipeline (without deprecated multi_class argument)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        solver='lbfgs', 
        max_iter=2000, 
        random_state=42
    ))
])

# 7. Split the Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 8. Define the Hyperparameter Grid
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100]
}

# 9. Execute Grid Search with Cross-Validation
print("Starting Grid Search...")
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# 10. Extract Best Model and Evaluate
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n--- Model Optimization Results ---")
print(f"Best Regularization Parameter (C): {grid_search.best_params_['classifier__C']}")
print(f"Test Set Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

print("--- Classification Report ---")
target_names = le.inverse_transform(range(len(le.classes_)))
print(classification_report(y_test, y_pred, target_names=target_names))

Starting Grid Search...

--- Model Optimization Results ---
Best Regularization Parameter (C): 100
Test Set Accuracy: 95.98%

--- Classification Report ---
                     precision    recall  f1-score   support

Insufficient_Weight       0.96      1.00      0.98        54
      Normal_Weight       0.93      0.91      0.92        58
     Obesity_Type_I       0.97      0.97      0.97        70
    Obesity_Type_II       0.95      0.98      0.97        60
   Obesity_Type_III       1.00      0.98      0.99        65
 Overweight_Level_I       0.93      0.91      0.92        58
Overweight_Level_II       0.96      0.95      0.96        58

           accuracy                           0.96       423
          macro avg       0.96      0.96      0.96       423
       weighted avg       0.96      0.96      0.96       423



gridsearch best parameters find kor e 
each c value is trained and tested 5 times
but sometimes those paramateres may overfit so the 5 fold cross validation is done 